In [ ]:
from pyspark.sql import SparkSession
import pandas as pd

# Inicializa a sessão do PySpark configurada para conectar com o Elasticsearch
spark = SparkSession.builder \
    .appName("EDA_Limpeza") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()


# Definir caminhos no container
caminho_cnefe = "/app/data/bruto/29_BA.csv"
caminho_busca = "/app/data/bruto/2026_teste_tecnico_relinke_cidacs.xlsx"

# Leitura do CNEFE 
df_cnefe = spark.read.csv(caminho_cnefe, header=True, inferSchema=True, sep=";")

# Leitura da Base de Busca 
df_busca_pd = pd.read_excel(caminho_busca).astype(str)
df_busca = spark.createDataFrame(df_busca_pd)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/10 05:53:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/10 05:53:44 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [ ]:
print("ESTRUTURA DO CNEFE")
df_cnefe.printSchema()

print("\nESTRUTURA DA BASE DE BUSCA")
df_busca.printSchema()

=== ESTRUTURA DO CNEFE ===
root
 |-- COD_UNICO_ENDERECO: integer (nullable = true)
 |-- COD_UF: integer (nullable = true)
 |-- COD_MUNICIPIO: integer (nullable = true)
 |-- COD_DISTRITO: integer (nullable = true)
 |-- COD_SUBDISTRITO: long (nullable = true)
 |-- COD_SETOR: string (nullable = true)
 |-- NUM_QUADRA: integer (nullable = true)
 |-- NUM_FACE: integer (nullable = true)
 |-- CEP: integer (nullable = true)
 |-- DSC_LOCALIDADE: string (nullable = true)
 |-- NOM_TIPO_SEGLOGR: string (nullable = true)
 |-- NOM_TITULO_SEGLOGR: string (nullable = true)
 |-- NOM_SEGLOGR: string (nullable = true)
 |-- NUM_ENDERECO: integer (nullable = true)
 |-- DSC_MODIFICADOR: string (nullable = true)
 |-- NOM_COMP_ELEM1: string (nullable = true)
 |-- VAL_COMP_ELEM1: string (nullable = true)
 |-- NOM_COMP_ELEM2: string (nullable = true)
 |-- VAL_COMP_ELEM2: string (nullable = true)
 |-- NOM_COMP_ELEM3: string (nullable = true)
 |-- VAL_COMP_ELEM3: string (nullable = true)
 |-- NOM_COMP_ELEM4: strin

In [3]:
print("Amostra CNEFE:")
df_cnefe.show(5, truncate=False)

print("Amostra Base de Busca:")
df_busca.show(5, truncate=False)

Amostra CNEFE:


26/10/10 05:53:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------+------+-------------+------------+---------------+----------------+----------+--------+--------+--------------+----------------+------------------+-----------------+------------+---------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+--------------+----------+----------+------------+-----------+-------------------+----------------------------+----------------------------+------------------------------+---------------+
|COD_UNICO_ENDERECO|COD_UF|COD_MUNICIPIO|COD_DISTRITO|COD_SUBDISTRITO|COD_SETOR       |NUM_QUADRA|NUM_FACE|CEP     |DSC_LOCALIDADE|NOM_TIPO_SEGLOGR|NOM_TITULO_SEGLOGR|NOM_SEGLOGR      |NUM_ENDERECO|DSC_MODIFICADOR|NOM_COMP_ELEM1|VAL_COMP_ELEM1|NOM_COMP_ELEM2|VAL_COMP_ELEM2|NOM_COMP_ELEM3|VAL_COMP_ELEM3|NOM_COMP_ELEM4|VAL_COMP_ELEM4|NOM_COMP_ELEM5|VAL_COMP_ELEM5|LATITUDE  |LONGITUDE |NV_GEO_COORD|COD_ESPECIE|DSC_ESTABELECIMENTO|COD_INDICADOR_ESTAB_ENDERECO|COD

In [ ]:
#Manter apenas colunas essenciais do CNEFE
colunas_cnefe_uteis = [
    "COD_UNICO_ENDERECO",
    "COD_SETOR",
    "COD_MUNICIPIO",
    "NOM_TIPO_SEGLOGR",
    "NOM_TITULO_SEGLOGR",
    "NOM_SEGLOGR",
    "NUM_ENDERECO",
    "DSC_LOCALIDADE",
    "CEP",
    "LATITUDE",
    "LONGITUDE"
]

#Manter apenas colunas essenciais da Base de Busca
colunas_busca_uteis = [
    "consulta_logradouro",
    "consulta_numero",
    "consulta_bairro",
    "consulta_complemento",
    "consulta_cep",
    "consulta_municipio"
]

df_cnefe_filtrado = df_cnefe.select(*colunas_cnefe_uteis)
df_busca_filtrado = df_busca.select(*colunas_busca_uteis)

In [ ]:
from pyspark.sql.functions import col, upper, trim, regexp_replace, translate, lpad, concat_ws, coalesce, lit, substring
from pyspark.sql.types import DoubleType

# Processamento e criação de colunas no CNEFE
df_cnefe_preparado = df_cnefe_filtrado \
    .withColumn(
        "logradouro_completo",
        trim(concat_ws(" ", 
                       coalesce(col("NOM_TIPO_SEGLOGR"), lit("")), 
                       coalesce(col("NOM_TITULO_SEGLOGR"), lit("")), 
                       coalesce(col("NOM_SEGLOGR"), lit(""))))
    ) \
    .withColumn("cep_limpo", lpad(regexp_replace(col("CEP").cast("string"), r"\D", ""), 8, "0")) \
    .withColumn("municipio_6digitos", substring(col("COD_MUNICIPIO").cast("string"), 1, 6)) \
    .withColumn("numero_endereco_limpo", regexp_replace(col("NUM_ENDERECO").cast("string"), r"^0$", "SN")) \
    .withColumn("latitude_limpa", regexp_replace(regexp_replace(col("LATITUDE").cast("string"), " ", "."), ",", ".").cast(DoubleType())) \
    .withColumn("longitude_limpa", regexp_replace(regexp_replace(col("LONGITUDE").cast("string"), " ", "."), ",", ".").cast(DoubleType()))

# Processamento da Base de Busca
df_busca_preparada = df_busca_filtrado \
    .withColumn("cep_limpo", lpad(regexp_replace(col("consulta_cep"), r"\D", ""), 8, "0")) \
    .withColumn("municipio_6digitos", substring(col("consulta_municipio").cast("string"), 1, 6))

# Limpar colunas textuais 
def limpar_dataframe_texto(df, colunas_texto):
    acentos = "ÁÀÃÂÉÈÊÍÌÓÒÕÔÚÙÜÇ"
    sem_acentos = "AAAAEEEIIOOOOUUUC"
    
    for c in colunas_texto:
        df = df.withColumn(c, regexp_replace(col(c).cast("string"), r"^nan$|^$", ""))
        df = df.withColumn(c, upper(col(c)))
        df = df.withColumn(c, translate(col(c), acentos, sem_acentos))
        df = df.withColumn(c, regexp_replace(col(c), r"[^A-Z0-9\s]", " "))
        df = df.withColumn(c, trim(regexp_replace(col(c), r"\s+", " ")))
    return df

# Aplicar a limpeza nas colunas de texto
cols_texto_cnefe = ["logradouro_completo", "numero_endereco_limpo", "DSC_LOCALIDADE"]
cols_texto_busca = ["consulta_logradouro", "consulta_numero", "consulta_bairro", "consulta_complemento"]

df_cnefe_limpo = limpar_dataframe_texto(df_cnefe_preparado, cols_texto_cnefe)
df_busca_limpo = limpar_dataframe_texto(df_busca_preparada, cols_texto_busca)

# Seleção final das colunas organizadas com os tipos e nomes padronizados
df_cnefe_final = df_cnefe_limpo.select(
    col("COD_UNICO_ENDERECO").cast("string").alias("codigo_unico_endereco"),
    col("COD_SETOR").cast("string").alias("codigo_setor_censitario"),
    col("municipio_6digitos").alias("codigo_municipio"),
    col("logradouro_completo").alias("logradouro"),
    col("numero_endereco_limpo").alias("numero"),
    col("DSC_LOCALIDADE").alias("bairro"),
    col("cep_limpo").alias("cep"),
    col("latitude_limpa").alias("latitude"),
    col("longitude_limpa").alias("longitude")
)

df_busca_final = df_busca_limpo.select(
    col("consulta_logradouro").alias("logradouro"),
    col("consulta_numero").alias("numero"),
    col("consulta_bairro").alias("bairro"),
    col("consulta_complemento").alias("complemento"),
    col("cep_limpo").alias("cep"),
    col("municipio_6digitos").alias("codigo_municipio")
)

print("CNEFE FINAL (AMOSTRA)")
df_cnefe_final.show(5, truncate=False)

print("BASE DE BUSCA FINAL (AMOSTRA)")
df_busca_final.show(5, truncate=False)

=== CNEFE FINAL (AMOSTRA) ===
+---------------------+-----------------------+----------------+------------------------------+------+--------------+--------+----------+----------+
|codigo_unico_endereco|codigo_setor_censitario|codigo_municipio|logradouro                    |numero|bairro        |cep     |latitude  |longitude |
+---------------------+-----------------------+----------------+------------------------------+------+--------------+--------+----------+----------+
|112026678            |290080125000007P       |290080          |BAIXA SEM DENOMINACAO         |SN    |POUSO ALEGRE  |45910000|-17.421301|-39.588591|
|112026678            |290080125000007P       |290080          |BAIXA SEM DENOMINACAO         |SN    |POUSO ALEGRE  |45910000|-17.421301|-39.588591|
|192615207            |290100705000002P       |290100          |AVENIDA DOUTOR ABELARDO VELOSO|776   |AMARGOSA      |45300000|-13.03028 |-39.595209|
|24076305             |290100705000029P       |290100          |POVOADO LAGO

In [ ]:
import os

diretorio_saida = "/app/data/processado"
os.makedirs(diretorio_saida, exist_ok=True)

#Exportar bases limpas em formato Parquet 
df_cnefe_final.write.mode("overwrite").parquet(f"{diretorio_saida}/cnefe_limpo.parquet")
df_busca_final.write.mode("overwrite").parquet(f"{diretorio_saida}/busca_limpa.parquet")

#Exportar a base de busca em CSV 
df_busca_final.coalesce(1).write.mode("overwrite").option("header", "true").csv(f"{diretorio_saida}/busca_limpa_csv")

#Mover o CSV gerado pelo Spark 
pasta_csv = f"{diretorio_saida}/busca_limpa_csv"
arquivo_gerado = [f for f in os.listdir(pasta_csv) if f.endswith(".csv")][0]
os.rename(os.path.join(pasta_csv, arquivo_gerado), f"{diretorio_saida}/busca_limpa.csv")

print("Arquivos Parquet e CSV exportados com sucesso para /app/data/processado!")

✓ Arquivos Parquet e CSV exportados com sucesso para /app/data/processado!
